## 05 — Export & Evaluate

### 0. Setup

In [1]:
# Core imports
import shutil
import zipfile
from pathlib import Path

# -- Project root --------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = Path("..") if NOTEBOOK_DIR.name == "notebooks" else Path(".")
print(f"Project root: {PROJECT_ROOT}")

# -- Paths (all under data/) ---------------------------------------------------
DATA_DIR       = PROJECT_ROOT / "data"
TRAINING_DIR   = DATA_DIR / "noodles_training"
FINETUNE_DIR   = DATA_DIR / "noodles_finetune_dataset"
# Synthetic dataset lives in yolo_dataset/ at the project root (emitted by
# blender_yolo_generator_v2.py and consumed by notebook 02).
DATASET_DIR    = PROJECT_ROOT / "yolo_dataset"
MODELS_DIR     = PROJECT_ROOT / "models" / "piece_segmentor"

# -- Shared constants (14 classes: 0-10 pieces, 11 board, 12 hinge, 13 pin) ---
PIECE_LABELS     = list("ABCDEFGHIJK")                              # piece classes 0-10
ALL_CLASS_NAMES  = PIECE_LABELS + ["board", "hinge", "pin"]         # classes 11, 12, 13
NUM_CLASSES      = len(ALL_CLASS_NAMES)                             # 14

PIECE_COLORS = {
    'A': ('Yellow',      (0xF9, 0xD6, 0x5E)),
    'B': ('SkyBlue',     (0x08, 0xA7, 0xE8)),
    'C': ('DarkBlue',    (0x20, 0x6D, 0xD9)),
    'D': ('Green',       (0x1F, 0xA1, 0x5B)),
    'E': ('Red',         (0xEE, 0x39, 0x4F)),
    'F': ('Teal',        (0x85, 0xDA, 0xBB)),
    'G': ('Pink',        (0xEC, 0x71, 0xA8)),
    'H': ('Purple',      (0xC7, 0x78, 0xB9)),
    'I': ('Orange',      (0xFC, 0x69, 0x0C)),
    'J': ('DarkRed',     (0xB6, 0x30, 0x48)),
    'K': ('YellowGreen', (0x95, 0xD4, 0x50)),
}

print(f"  Training dir: {TRAINING_DIR}")
print(f"  Models dir:   {MODELS_DIR}")


Project root: ..
  Training dir: ..\data\noodles_training
  Models dir:   ..\models\piece_segmentor


### 1. Find Best Model

In [2]:
# Prefer Phase 2 best.pt → Phase 1 best.pt → Phase 2 last.pt → Phase 1 last.pt
candidates = [
    TRAINING_DIR / 'phase2_finetune'   / 'weights' / 'best.pt',
    TRAINING_DIR / 'phase1_synthetic'  / 'weights' / 'best.pt',
    TRAINING_DIR / 'phase2_finetune'   / 'weights' / 'last.pt',
    TRAINING_DIR / 'phase1_synthetic'  / 'weights' / 'last.pt',
]

best_model_path = None
for c in candidates:
    if c.exists():
        best_model_path = c
        break

if best_model_path:
    print(f"✅ Best model: {best_model_path}")
    print(f"   Size: {best_model_path.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("❌ No model weights found! Run notebooks 02/04 first.")

✅ Best model: ..\data\noodles_training\phase2_finetune\weights\best.pt
   Size: 6.2 MB


### 2. Export to TorchScript

In [3]:
from ultralytics import YOLO

assert best_model_path and best_model_path.exists(), "No model to export!"

model = YOLO(str(best_model_path))
export_path = model.export(format='torchscript')

print(f"\n✅ Exported: {export_path}")
print(f"   Size: {Path(export_path).stat().st_size / 1024 / 1024:.1f} MB")

Ultralytics 8.4.39  Python-3.11.9 torch-2.11.0+cu128 CPU (11th Gen Intel Core i7-11700K @ 3.60GHz)
YOLO26n-seg summary (fused): 139 layers, 2,691,614 parameters, 0 gradients, 9.0 GFLOPs

PyTorch: starting from '..\data\noodles_training\phase2_finetune\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 300, 38), (1, 32, 160, 160)) (6.2 MB)

TorchScript: starting export with torch 2.11.0+cu128...
TorchScript: export success  3.9s, saved as '..\data\noodles_training\phase2_finetune\weights\best.torchscript' (11.0 MB)

Export complete (4.4s)
Results saved to C:\Users\abdul\Desktop\noodels\data\noodles_training\phase2_finetune\weights
Predict:         yolo predict task=segment model=..\data\noodles_training\phase2_finetune\weights\best.torchscript imgsz=640 
Validate:        yolo val task=segment model=..\data\noodles_training\phase2_finetune\weights\best.torchscript imgsz=640 data=C:\Users\abdul\Desktop\noodels\data\noodles_finetune_dataset\dataset.yaml  
Visu

In [4]:
# Export to ONNX format
onnx_export_path = model.export(
    format='onnx',
    dynamic=False,      # Static input size
    simplify=True,      # Simplify the graph
    imgsz=640,          # Input size (must match your webapp)
    opset=11            # ONNX opset version for browser compatibility
)

print(f"\n✅ Exported ONNX: {onnx_export_path}")
print(f"   Size: {Path(onnx_export_path).stat().st_size / 1024 / 1024:.1f} MB")

Ultralytics 8.4.39  Python-3.11.9 torch-2.11.0+cu128 CPU (11th Gen Intel Core i7-11700K @ 3.60GHz)
YOLO26n-seg summary (fused): 139 layers, 2,691,614 parameters, 0 gradients, 9.0 GFLOPs

PyTorch: starting from '..\data\noodles_training\phase2_finetune\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 300, 38), (1, 32, 160, 160)) (6.2 MB)

ONNX: starting export with onnx 1.21.0 opset 11...


c:\Users\abdul\Desktop\noodels\.venv\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 11 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.91...
ONNX: export success  3.1s, saved as '..\data\noodles_training\phase2_finetune\weights\best.onnx' (10.6 MB)

Export complete (3.5s)
Results saved to C:\Users\abdul\Desktop\noodels\data\noodles_training\phase2_finetune\weights
Predict:         yolo predict task=segment model=..\data\noodles_training\phase2_finetune\weights\best.onnx imgsz=640 
Validate:        yolo val task=segment model=..\data\noodles_training\phase2_finetune\weights\best.onnx imgsz=640 data=C:\Users\abdul\Desktop\noodels\data\noodles_finetune_dataset\dataset.yaml  
Visualize:       https://netron.app

✅ Exported ONNX: ..\data\noodles_training\phase2_finetune\weights\best.onnx
   Size: 10.6 MB


### 3. Package Model

In [5]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Determine phase directory
p2_dir = TRAINING_DIR / 'phase2_finetune'
p1_dir = TRAINING_DIR / 'phase1_synthetic'
phase_dir = p2_dir if p2_dir.exists() else p1_dir

zip_path = MODELS_DIR / 'noodles_yolo_seg_trained.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Add best/last weights
    for wt in ['best.pt', 'last.pt', 'best.torchscript', 'best.onnx']:
        wt_path = phase_dir / 'weights' / wt
        if wt_path.exists():
            zf.write(wt_path, f"weights/{wt}")
            print(f"  + weights/{wt}")
    
    # Add results CSV and plots
    for fname in ['results.csv', 'results.png', 'confusion_matrix.png', 'labels.jpg']:
        fpath = phase_dir / fname
        if fpath.exists():
            zf.write(fpath, fname)
            print(f"  + {fname}")
    
    # Add Phase 1 weights too if Phase 2 exists
    if phase_dir == p2_dir and (p1_dir / 'weights' / 'best.pt').exists():
        p1_best = p1_dir / 'weights' / 'best.pt'
        zf.write(p1_best, 'phase1_weights/best.pt')
        print(f"  + phase1_weights/best.pt")
    
    # Add dataset configs
    for ds_yaml in [DATASET_DIR / 'dataset.yaml', FINETUNE_DIR / 'dataset.yaml']:
        if ds_yaml.exists():
            arcname = f"configs/{ds_yaml.parent.name}_{ds_yaml.name}"
            zf.write(ds_yaml, arcname)
            print(f"  + {arcname}")
    
    classes_file = DATASET_DIR / 'classes.txt'
    if classes_file.exists():
        zf.write(classes_file, 'classes.txt')
        print(f"  + classes.txt")

print(f"\n✅ Packaged model: {zip_path}")
print(f"   Size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB")

  + weights/best.pt
  + weights/last.pt
  + weights/best.torchscript
  + weights/best.onnx
  + results.csv
  + results.png
  + confusion_matrix.png
  + labels.jpg
  + phase1_weights/best.pt
  + configs/yolo_dataset_dataset.yaml
  + configs/noodles_finetune_dataset_dataset.yaml

✅ Packaged model: ..\models\piece_segmentor\noodles_yolo_seg_trained.zip
   Size: 36.0 MB


### 4. Test on Local Photo (Optional)

In [6]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Set a test image path or None to skip
TEST_IMAGE = str(PROJECT_ROOT / '20260324_150837.jpg')  # e.g. str(PROJECT_ROOT / 'test_photo.jpg')

if TEST_IMAGE and Path(TEST_IMAGE).exists():
    model = YOLO(str(best_model_path))
    results = model.predict(source=TEST_IMAGE, imgsz=640, conf=0.3)
    
    # Draw results
    for r in results:
        annotated = r.plot()
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title("Test Photo Prediction")
        plt.axis('off')
        plt.show()
        
        if r.boxes is not None:
            print(f"\nDetected {len(r.boxes)} objects:")
            for box in r.boxes:
                cls_id = int(box.cls)
                conf = float(box.conf)
                label = ALL_CLASS_NAMES[cls_id] if cls_id < len(ALL_CLASS_NAMES) else f'cls{cls_id}'
                if cls_id < len(PIECE_LABELS):
                    color = PIECE_COLORS.get(label, ('?', (0,0,0)))[0]
                    print(f"  Piece {label} ({color}): conf={conf:.3f}")
                else:
                    print(f"  {label}: conf={conf:.3f}")
else:
    print("No test image specified. Set TEST_IMAGE to a path to test.")

No test image specified. Set TEST_IMAGE to a path to test.


### 5. Summary

In [7]:
print("="*60)
print("IQ Noodles — Training Pipeline Summary")
print("="*60)
print(f"\nProject root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

# Check what exists
items = [
    ('Synthetic dataset',  DATASET_DIR / 'dataset.yaml'),
    ('Real finetune data', FINETUNE_DIR / 'dataset.yaml'),
    ('Phase 1 weights',    TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'),
    ('Phase 2 weights',    TRAINING_DIR / 'phase2_finetune' / 'weights' / 'best.pt'),
    ('Exported model',     zip_path if 'zip_path' in dir() else Path('nonexistent')),
]

for label, path in items:
    status = '✅' if path.exists() else '❌'
    size = f"({path.stat().st_size / 1024 / 1024:.1f} MB)" if path.exists() and path.is_file() else ''
    print(f"  {status} {label}: {path.name} {size}")

print(f"\n{'='*60}")
print("Done! 🎉")

IQ Noodles — Training Pipeline Summary

Project root: ..
Data dir:     ..\data
  ✅ Synthetic dataset: dataset.yaml (0.0 MB)
  ✅ Real finetune data: dataset.yaml (0.0 MB)
  ✅ Phase 1 weights: best.pt (6.3 MB)
  ✅ Phase 2 weights: best.pt (6.2 MB)
  ✅ Exported model: noodles_yolo_seg_trained.zip (36.0 MB)

Done! 🎉


In [8]:
# ── Auto-deploy: copy best.onnx to webapp-v4 ──────────────────────────────────
import shutil
from pathlib import Path

ONNX_SRC  = TRAINING_DIR / "phase2_finetune" / "weights" / "best.onnx"
ONNX_DEST = PROJECT_ROOT / "webapp-v4" / "public" / "models" / "yolo26n-seg.onnx"

if not ONNX_SRC.exists():
    # fallback to phase1 if phase2 not available
    ONNX_SRC = TRAINING_DIR / "phase1_synthetic" / "weights" / "best.onnx"

if ONNX_SRC.exists():
    ONNX_DEST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ONNX_SRC, ONNX_DEST)
    print(f"✅ Deployed: {ONNX_SRC.name} → {ONNX_DEST}")
    print(f"   Size: {ONNX_DEST.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("⚠️  No ONNX model found. Run export step first.")


✅ Deployed: best.onnx → ..\webapp-v4\public\models\yolo26n-seg.onnx
   Size: 10.6 MB
